In [ ]:
!pip install yfinance pandas numpy faiss-cpu sentence-transformers google-generativeai

INFO: pip is looking at multiple versions of googleapis-common-protos to determine which version is compatible with other requirements. This could take a while.
INFO: pip is looking at multiple versions of proto-plus to determine which version is compatible with other requirements. This could take a while.
INFO: pip is looking at multiple versions of grpcio-status to determine which version is compatible with other requirements. This could take a while.
INFO: pip is still looking at multiple versions of grpcio-status to determine which version is compatible with other requirements. This could take a while.
INFO: This is taking longer than usual. You might need to provide the dependency resolver with stricter constraints to reduce runtime. See https://pip.pypa.io/warnings/backtracking for guidance. If you want to abort this run, press Ctrl + C.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 36.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 155.1/155.1 kB 7.4 MB

In [ ]:
import yfinance as yf
import pandas as pd
import numpy as np


In [ ]:
ticker = yf.Ticker("AAPL")

In [ ]:
ticker.info

{'address1': 'One Apple Park Way',
 'city': 'Cupertino',
 'state': 'CA',
 'zip': '95014',
 'country': 'United States',
 'phone': '(408) 996-1010',
 'website': 'https://www.apple.com',
 'industry': 'Consumer Electronics',
 'industryKey': 'consumer-electronics',
 'industryDisp': 'Consumer Electronics',
 'sector': 'Technology',
 'sectorKey': 'technology',
 'sectorDisp': 'Technology',
 'longBusinessSummary': 'Apple Inc. designs, manufactures, and markets smartphones, personal computers, tablets, wearables, and accessories worldwide. The company offers iPhone, a line of smartphones; Mac, a line of personal computers; iPad, a line of multi-purpose tablets; and wearables, home, and accessories comprising AirPods, Apple Vision Pro, Apple TV, Apple Watch, Beats products, and HomePod, as well as Apple branded and third-party accessories. It also provides AppleCare support and cloud services; and operates various platforms, including the App Store that allows customers to discover and download ap

In [ ]:
print("Company Name:", ticker.info.get("longName"))
print("Sector:", ticker.info.get("sector"))
print("Industry:", ticker.info.get("industry"))

Company Name: Apple Inc.
Sector: Technology
Industry: Consumer Electronics


# **Collect Apple's financial statements**

In [ ]:
income_statement = ticker.income_stmt

income_statement.head()

,2025-09-30,2024-09-30,2023-09-30,2022-09-30
Tax Effect Of Unusual Items,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00
Tax Rate For Calcs,1.560000e-01,2.410000e-01,1.470000e-01,1.620000e-01
Normalized EBITDA,1.447480e+11,1.346610e+11,1.258200e+11,1.305410e+11
Net Income From Continuing Operation Net Minority Interest,1.120100e+11,9.373600e+10,9.699500e+10,9.980300e+10
Reconciled Depreciation,1.169800e+10,1.144500e+10,1.151900e+10,1.110400e+10


In [ ]:
balance_sheet = ticker.balance_sheet

balance_sheet.head()

,2025-09-30,2024-09-30,2023-09-30,2022-09-30
Treasury Shares Number,NaN,NaN,0.000000e+00,NaN
Ordinary Shares Number,1.477326e+10,1.511679e+10,1.555006e+10,1.594342e+10
Share Issued,1.477326e+10,1.511679e+10,1.555006e+10,1.594342e+10
Net Debt,6.272300e+10,7.668600e+10,8.112300e+10,9.642300e+10
Total Debt,9.865700e+10,1.066290e+11,1.110880e+11,1.324800e+11


In [ ]:
cash_flow = ticker.cashflow

cash_flow.head()

,2025-09-30,2024-09-30,2023-09-30,2022-09-30
Free Cash Flow,9.876700e+10,1.088070e+11,9.958400e+10,1.114430e+11
Repurchase Of Capital Stock,-9.071100e+10,-9.494900e+10,-7.755000e+10,-8.940200e+10
Repayment Of Debt,-1.093200e+10,-9.958000e+09,-1.115100e+10,-9.543000e+09
Issuance Of Debt,4.481000e+09,0.000000e+00,5.228000e+09,5.465000e+09
Capital Expenditure,-1.271500e+10,-9.447000e+09,-1.095900e+10,-1.070800e+10


In [ ]:
print("Income Statement:", income_statement.shape)
print("Balance Sheet:", balance_sheet.shape)
print("Cash Flow Statement:", cash_flow.shape)

Income Statement: (39, 4)
Balance Sheet: (69, 4)
Cash Flow Statement: (51, 4)


# **Collect historical stock prices and key financial metrics**

In [ ]:
historical_data = ticker.history(period="5y")

historical_data.head()

,Open,High,Low,Close,Volume,Dividends,Stock Splits
Date,,,,,,,
2021-10-11 00:00:00-04:00,138.766061,141.243497,138.317384,139.292755,64452200,0.0,0.0
2021-10-12 00:00:00-04:00,139.702374,139.721885,137.566309,138.024734,73035900,0.0,0.0
2021-10-13 00:00:00-04:00,137.761434,137.917482,135.771669,137.439560,78762700,0.0,0.0
2021-10-14 00:00:00-04:00,138.609974,140.336384,138.024745,140.219330,69907100,0.0,0.0
2021-10-15 00:00:00-04:00,140.229111,141.331271,139.975505,141.272751,67940300,0.0,0.0


In [ ]:
print("Historical data shape:", historical_data.shape)

Historical data shape: (1254, 7)


In [ ]:
info = ticker.info

metrics = {
    "Market Cap": info.get("marketCap"),
    "Current Price": info.get("currentPrice"),
    "Revenue": info.get("totalRevenue"),
    "Net Income": info.get("netIncomeToCommon"),
    "EPS": info.get("trailingEps"),
    "P/E Ratio": info.get("trailingPE"),
    "Price to Book": info.get("priceToBook"),
    "Return on Equity": info.get("returnOnEquity"),
    "Profit Margin": info.get("profitMargins"),
    "Operating Margin": info.get("operatingMargins"),
    "Debt to Equity": info.get("debtToEquity"),
    "Free Cash Flow": info.get("freeCashflow"),
}

metrics

{'Market Cap': 4968150990848,
 'Current Price': 340.42,
 'Revenue': 466822987776,
 'Net Income': 128929996800,
 'EPS': 8.72,
 'P/E Ratio': 39.03899,
 'Price to Book': 46.25272,
 'Return on Equity': 1.4875101,
 'Profit Margin': 0.27618998,
 'Operating Margin': 0.32623002,
 'Debt to Equity': 78.445,
 'Free Cash Flow': 107721875456}

In [ ]:
metrics_df = pd.DataFrame(
    metrics.items(),
    columns=["Metric", "Value"]
)

metrics_df

,Metric,Value
0,Market Cap,4.968151e+12
1,Current Price,3.404200e+02
2,Revenue,4.668230e+11
3,Net Income,1.289300e+11
4,EPS,8.720000e+00
5,P/E Ratio,3.903899e+01
6,Price to Book,4.625272e+01
7,Return on Equity,1.487510e+00
8,Profit Margin,2.761900e-01
9,Operating Margin,3.262300e-01


# **Convert the financial data into text.**

In [ ]:
income_text = income_statement.to_string()

print(income_text[:3000])

                                                              2025-09-30    2024-09-30    2023-09-30    2022-09-30
Tax Effect Of Unusual Items                                 0.000000e+00  0.000000e+00  0.000000e+00  0.000000e+00
Tax Rate For Calcs                                          1.560000e-01  2.410000e-01  1.470000e-01  1.620000e-01
Normalized EBITDA                                           1.447480e+11  1.346610e+11  1.258200e+11  1.305410e+11
Net Income From Continuing Operation Net Minority Interest  1.120100e+11  9.373600e+10  9.699500e+10  9.980300e+10
Reconciled Depreciation                                     1.169800e+10  1.144500e+10  1.151900e+10  1.110400e+10
Reconciled Cost Of Revenue                                  2.209600e+11  2.103520e+11  2.141370e+11  2.235460e+11
EBITDA                                                      1.447480e+11  1.346610e+11  1.258200e+11  1.305410e+11
EBIT                                                        1.330500e+11  1.2321

In [ ]:
balance_text = balance_sheet.to_string()

print(balance_text[:3000])

                                                       2025-09-30    2024-09-30    2023-09-30    2022-09-30
Treasury Shares Number                                        NaN           NaN  0.000000e+00           NaN
Ordinary Shares Number                               1.477326e+10  1.511679e+10  1.555006e+10  1.594342e+10
Share Issued                                         1.477326e+10  1.511679e+10  1.555006e+10  1.594342e+10
Net Debt                                             6.272300e+10  7.668600e+10  8.112300e+10  9.642300e+10
Total Debt                                           9.865700e+10  1.066290e+11  1.110880e+11  1.324800e+11
Tangible Book Value                                  7.373300e+10  5.695000e+10  6.214600e+10  5.067200e+10
Invested Capital                                     1.723900e+11  1.635790e+11  1.732340e+11  1.707410e+11
Working Capital                                     -1.767400e+10 -2.340500e+10 -1.742000e+09 -1.857700e+10
Net Tangible Assets         

In [ ]:
cashflow_text = cash_flow.to_string()

print(cashflow_text[:3000])

                                                  2025-09-30    2024-09-30    2023-09-30    2022-09-30
Free Cash Flow                                  9.876700e+10  1.088070e+11  9.958400e+10  1.114430e+11
Repurchase Of Capital Stock                    -9.071100e+10 -9.494900e+10 -7.755000e+10 -8.940200e+10
Repayment Of Debt                              -1.093200e+10 -9.958000e+09 -1.115100e+10 -9.543000e+09
Issuance Of Debt                                4.481000e+09  0.000000e+00  5.228000e+09  5.465000e+09
Capital Expenditure                            -1.271500e+10 -9.447000e+09 -1.095900e+10 -1.070800e+10
Interest Paid Supplemental Data                          NaN           NaN  3.803000e+09  2.865000e+09
Income Tax Paid Supplemental Data               4.336900e+10  2.610200e+10  1.867900e+10  1.957300e+10
End Cash Position                               3.593400e+10  2.994300e+10  3.073700e+10  2.497700e+10
Beginning Cash Position                         2.994300e+10  3.073700e+1

In [ ]:
historical_summary = historical_data.describe().to_string()

print(historical_summary)

              Open         High          Low        Close        Volume    Dividends  Stock Splits
count  1254.000000  1254.000000  1254.000000  1254.000000  1.254000e+03  1254.000000        1254.0
mean    203.315315   205.578726   201.312992   203.552578  6.335963e+07     0.003907           0.0
std      51.790739    52.277757    51.381060    51.870949  2.858368e+07     0.030764           0.0
min     123.800267   125.529397   121.992536   122.827621  1.791060e+07     0.000000           0.0
25%     163.719221   165.466764   162.345993   163.650394  4.470120e+07     0.000000           0.0
50%     189.140595   191.097134   187.890451   189.338287  5.486660e+07     0.000000           0.0
75%     235.620123   238.452066   232.682001   235.871006  7.491310e+07     0.000000           0.0
max     341.079987   345.339996   338.750000   341.070007  3.186799e+08     0.270000           0.0


In [ ]:
metrics_text = metrics_df.to_string(index=False)

print(metrics_text)

          Metric        Value
      Market Cap 4.968151e+12
   Current Price 3.404200e+02
         Revenue 4.668230e+11
      Net Income 1.289300e+11
             EPS 8.720000e+00
       P/E Ratio 3.903899e+01
   Price to Book 4.625272e+01
Return on Equity 1.487510e+00
   Profit Margin 2.761900e-01
Operating Margin 3.262300e-01
  Debt to Equity 7.844500e+01
  Free Cash Flow 1.077219e+11


# **Create our Financial Knowledge Base**

In [ ]:
financial_documents = []

financial_documents.append({
    "title": "Apple Income Statement",
    "content": income_text
})

financial_documents.append({
    "title": "Apple Balance Sheet",
    "content": balance_text
})

financial_documents.append({
    "title": "Apple Cash Flow Statement",
    "content": cashflow_text
})

financial_documents.append({
    "title": "Apple Historical Performance",
    "content": historical_summary
})

financial_documents.append({
    "title": "Apple Financial Metrics",
    "content": metrics_text
})

In [ ]:
print("Number of documents:", len(financial_documents))

Number of documents: 5


In [ ]:
print(financial_documents[0]["title"])
print(financial_documents[0]["content"][:2000])

Apple Income Statement
                                                              2025-09-30    2024-09-30    2023-09-30    2022-09-30
Tax Effect Of Unusual Items                                 0.000000e+00  0.000000e+00  0.000000e+00  0.000000e+00
Tax Rate For Calcs                                          1.560000e-01  2.410000e-01  1.470000e-01  1.620000e-01
Normalized EBITDA                                           1.447480e+11  1.346610e+11  1.258200e+11  1.305410e+11
Net Income From Continuing Operation Net Minority Interest  1.120100e+11  9.373600e+10  9.699500e+10  9.980300e+10
Reconciled Depreciation                                     1.169800e+10  1.144500e+10  1.151900e+10  1.110400e+10
Reconciled Cost Of Revenue                                  2.209600e+11  2.103520e+11  2.141370e+11  2.235460e+11
EBITDA                                                      1.447480e+11  1.346610e+11  1.258200e+11  1.305410e+11
EBIT                                                     

# **Text Chunking for the RAG Knowledge Base**

In [ ]:
!pip install langchain


In [ ]:
!pip install -U langchain-text-splitters

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [ ]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200
)

In [ ]:
all_text = ""

for document in financial_documents:
    all_text += document["title"] + "\n"
    all_text += document["content"] + "\n\n"

In [ ]:
chunks = text_splitter.split_text(all_text)

In [ ]:
print("Number of chunks:", len(chunks))

Number of chunks: 24


In [ ]:
print(chunks[20])

Change In Payable                               9.020000e+08  6.020000e+09 -1.889000e+09  9.448000e+09
Change In Account Payable                       9.020000e+08  6.020000e+09 -1.889000e+09  9.448000e+09
Change In Inventory                             1.400000e+09 -1.046000e+09 -1.618000e+09  1.484000e+09
Change In Receivables                          -7.029000e+09 -5.144000e+09 -4.170000e+08 -9.343000e+09
Changes In Account Receivables                 -6.682000e+09 -3.788000e+09 -1.688000e+09 -1.823000e+09
Other Non Cash Items                           -8.900000e+07 -2.266000e+09 -2.227000e+09  1.006000e+09
Stock Based Compensation                        1.286300e+10  1.168800e+10  1.083300e+10  9.038000e+09
Deferred Tax                                             NaN           NaN           NaN  8.950000e+08
Deferred Income Tax                                      NaN           NaN           NaN  8.950000e+08


In [ ]:
print("Number of chunks:", len(chunks))

Number of chunks: 24


# **Generate embeddings**

In [ ]:
!pip install -U sentence-transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 740.6/740.6 kB 37.1 MB/s eta 0:00:00
  Attempting uninstall: sentence-transformers
    Found existing installation: sentence-transformers 5.7.0
    Uninstalling sentence-transformers-5.7.0:
      Successfully uninstalled sentence-transformers-5.7.0


In [ ]:
from sentence_transformers import SentenceTransformer

In [ ]:
embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

In [ ]:
embeddings = embedding_model.encode(chunks)

In [ ]:
print("Number of chunks:", len(chunks))
print("Embedding shape:", embeddings.shape)

Number of chunks: 24
Embedding shape: (24, 384)


# **Creating the FAISS vector database**

In [ ]:
!pip install faiss-cpu

In [ ]:
import faiss

In [ ]:
dimension = embeddings.shape[1]

index = faiss.IndexFlatL2(dimension)

In [ ]:
index.add(embeddings.astype("float32"))

In [ ]:
print("Number of vectors in FAISS:", index.ntotal)

Number of vectors in FAISS: 24


# **Search FAISS**

In [ ]:
question = "What is Apple's net income?"

In [ ]:
question_embedding = embedding_model.encode([question])

In [ ]:
distances, indices = index.search(
    question_embedding.astype("float32"),
    k=3
)

In [ ]:
for i in indices[0]:
    print("----- Retrieved Chunk -----")
    print(chunks[i])

----- Retrieved Chunk -----
Apple Financial Metrics
          Metric        Value
      Market Cap 4.913423e+12
   Current Price 3.366700e+02
         Revenue 4.668230e+11
      Net Income 1.289300e+11
             EPS 8.720000e+00
       P/E Ratio 3.860894e+01
   Price to Book 4.574321e+01
Return on Equity 1.487510e+00
   Profit Margin 2.761900e-01
Operating Margin 3.262300e-01
  Debt to Equity 7.844500e+01
  Free Cash Flow 1.077219e+11
----- Retrieved Chunk -----
Apple Income Statement
                                                              2025-09-30    2024-09-30    2023-09-30    2022-09-30
Tax Effect Of Unusual Items                                 0.000000e+00  0.000000e+00  0.000000e+00  0.000000e+00
Tax Rate For Calcs                                          1.560000e-01  2.410000e-01  1.470000e-01  1.620000e-01
Normalized EBITDA                                           1.447480e+11  1.346610e+11  1.258200e+11  1.305410e+11
Net Income From Continuing Operation Net Minori

# **Connect Gemini**

In [ ]:
!pip install -U google-generativeai

import google.generativeai as genai

INFO: pip is looking at multiple versions of googleapis-common-protos to determine which version is compatible with other requirements. This could take a while.
INFO: pip is looking at multiple versions of proto-plus to determine which version is compatible with other requirements. This could take a while.
INFO: pip is looking at multiple versions of grpcio-status to determine which version is compatible with other requirements. This could take a while.
INFO: pip is still looking at multiple versions of grpcio-status to determine which version is compatible with other requirements. This could take a while.
INFO: This is taking longer than usual. You might need to provide the dependency resolver with stricter constraints to reduce runtime. See https://pip.pypa.io/warnings/backtracking for guidance. If you want to abort this run, press Ctrl + C.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 155.1/155.1 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 21.3 MB/s

/tmp/ipykernel_1056/2204206780.py:3: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  import google.generativeai as genai


In [ ]:
from google.colab import userdata
import google.generativeai as genai

api_key = userdata.get("GEMINI_API_KEY")
genai.configure(api_key=api_key)

In [ ]:
model = genai.GenerativeModel("gemini-3.8-flash")

In [ ]:
response = model.generate_content("What is Apple's net income?")
print(response.text)

For its most recent full fiscal year (**Fiscal Year 2024**, ended September 28, 2024), Apple reported a net income of **$93.74 billion** (down slightly from $96.99 billion in FY 2023, largely due to a one-time $10.2 billion European tax charge).

### Recent Breakdown:
* **Fiscal Year 2024 (Full Year):** $93.74 billion
* **Fiscal Year 2023 (Full Year):** $96.995 billion
* **Most Recent Quarter (Q1 FY25, ended Dec 28, 2024):** $36.33 billion


# **Combine FAISS + Gemini**

In [ ]:
question = "What is Apple's net income?"

question_embedding = embedding_model.encode([question])

distances, indices = index.search(
    question_embedding.astype("float32"),
    k=3
)

retrieved_context = ""

for i in indices[0]:
    retrieved_context += chunks[i] + "\n\n"

prompt = f"""
Answer the question using ONLY the financial information provided below.

Financial Information:
{retrieved_context}

Question:
{question}

If the information is not available in the provided financial information, say:
"Information not available in the provided data."

Do not use outside information.
"""

response = model.generate_content(prompt)

print(response.text)

Based on the provided financial information:

* Under **Apple Financial Metrics**, Apple's **Net Income** is **1.289300e+11** (or approximately $128.93 billion).

Additionally, under the **Apple Income Statement**, **Net Income From Continuing Operation Net Minority Interest** is reported as:
* **2025-09-30:** 1.120100e+11
* **2024-09-30:** 9.373600e+10
* **2023-09-30:** 9.699500e+10
* **2022-09-30:** 9.980300e+10


In [ ]:
question = "What is Apple's revenue?"

question_embedding = embedding_model.encode([question])

distances, indices = index.search(
    question_embedding.astype("float32"),
    k=3
)

retrieved_context = ""

for i in indices[0]:
    retrieved_context += chunks[i] + "\n\n"

prompt = f"""
Answer the question using ONLY the financial information provided below.

Financial Information:
{retrieved_context}

Question:
{question}

If the information is not available in the provided financial information, say:
"Information not available in the provided data."

Do not use outside information.
"""

response = model.generate_content(prompt)

print(response.text)

Based on the provided financial information, Apple's revenue is **4.668230e+11** (or $466,823,000,000).


In [ ]:
question = "What is Apple's EPS?"

question_embedding = embedding_model.encode([question])

distances, indices = index.search(
    question_embedding.astype("float32"),
    k=3
)

retrieved_context = ""

for i in indices[0]:
    retrieved_context += chunks[i] + "\n\n"

prompt = f"""
Answer the question using ONLY the financial information provided below.

Financial Information:
{retrieved_context}

Question:
{question}

If the information is not available in the provided financial information, say:
"Information not available in the provided data."

Do not use outside information.
"""

response = model.generate_content(prompt)

print(response.text)

Based on the provided financial information, Apple's EPS is **8.72** (or 8.720000e+00).


In [ ]:
question = "What is Apple's debt and cash position?"

question_embedding = embedding_model.encode([question])

distances, indices = index.search(
    question_embedding.astype("float32"),
    k=3
)

retrieved_context = ""

for i in indices[0]:
    retrieved_context += chunks[i] + "\n\n"

prompt = f"""
Answer the question using ONLY the financial information provided below.

Financial Information:
{retrieved_context}

Question:
{question}

If the information is not available in the provided financial information, say:
"Information not available in the provided data."

Do not use outside information.
"""

response = model.generate_content(prompt)

print(response.text)

Based on the provided financial information, Apple's debt and cash positions across the reported periods are as follows:

### **Cash Position (End Cash Position)**
* **2025-09-30:** 3.593400e+10 (~$35.93 billion)
* **2024-09-30:** 2.994300e+10 (~$29.94 billion)
* **2023-09-30:** 3.073700e+10 (~$30.74 billion)
* **2022-09-30:** 2.497700e+10 (~$24.98 billion)

---

### **Debt Position**
* **Total Debt:**
  * **2025-09-30:** 9.865700e+10 (~$98.66 billion)
  * **2024-09-30:** 1.066290e+11 (~$106.63 billion)
  * **2023-09-30:** 1.110880e+11 (~$111.09 billion)
  * **2022-09-30:** 1.324800e+11 (~$132.48 billion)

* **Net Debt:**
  * **2025-09-30:** 6.272300e+10 (~$62.72 billion)
  * **2024-09-30:** 7.668600e+10 (~$76.69 billion)
  * **2023-09-30:** 8.112300e+10 (~$81.12 billion)
  * **2022-09-30:** 9.642300e+10 (~$96.42 billion)

* **Debt to Equity Ratio:** 7.844500e+01 (78.445)


In [ ]:
question = "What is Apple's free cash flow?"

question_embedding = embedding_model.encode([question])

distances, indices = index.search(
    question_embedding.astype("float32"),
    k=3
)

retrieved_context = ""

for i in indices[0]:
    retrieved_context += chunks[i] + "\n\n"

prompt = f"""
Answer the question using ONLY the financial information provided below.

Financial Information:
{retrieved_context}

Question:
{question}

If the information is not available in the provided financial information, say:
"Information not available in the provided data."

Do not use outside information.
"""

response = model.generate_content(prompt)

print(response.text)

Based on the provided financial information, Apple's Free Cash Flow is reported as follows:

* **Under Apple Financial Metrics:**
  * **Free Cash Flow:** 1.077219e+11 (or approximately $107.72 billion)

* **Under Apple Cash Flow Statement (by fiscal year):**
  * **2025-09-30:** 9.876700e+10 (or approximately $98.77 billion)
  * **2024-09-30:** 1.088070e+11 (or approximately $108.81 billion)
  * **2023-09-30:** 9.958400e+10 (or approximately $99.58 billion)
  * **2022-09-30:** 1.114430e+11 (or approximately $111.44 billion)


# **Test Financial Ratios**

In [ ]:
question = "What are Apple's financial ratios, including P/E ratio, Price-to-Book ratio, Return on Equity, Profit Margin, and Operating Margin?"

question_embedding = embedding_model.encode([question])

distances, indices = index.search(
    question_embedding.astype("float32"),
    k=3
)

retrieved_context = ""

for i in indices[0]:
    retrieved_context += chunks[i] + "\n\n"

prompt = f"""
Answer the question using ONLY the financial information provided below.

Financial Information:
{retrieved_context}

Question:
{question}

If the information is not available in the provided financial information, say:
"Information not available in the provided data."

Do not use outside information.
"""

response = model.generate_content(prompt)

print(response.text)

Based on the provided financial information, Apple's financial ratios are:

* **P/E Ratio:** 3.903899e+01 (or approximately 39.04)
* **Price to Book:** 4.625272e+01 (or approximately 46.25)
* **Return on Equity:** 1.487510e+00 (or approximately 1.488 / 148.75%)
* **Profit Margin:** 2.761900e-01 (or approximately 0.276 / 27.62%)
* **Operating Margin:** 3.262300e-01 (or approximately 0.326 / 32.62%)


In [ ]:
question = "What are Apple's total assets, total liabilities, and total debt?"

question_embedding = embedding_model.encode([question])

distances, indices = index.search(
    question_embedding.astype("float32"),
    k=3
)

retrieved_context = ""

for i in indices[0]:
    retrieved_context += chunks[i] + "\n\n"

prompt = f"""
Answer the question using ONLY the financial information provided below.

Financial Information:
{retrieved_context}

Question:
{question}

If the information is not available in the provided financial information, say:
"Information not available in the provided data."

Do not use outside information.
"""

response = model.generate_content(prompt)

print(response.text)

Based on the provided financial information:

* **Total Assets:** Information not available in the provided data.
* **Total Liabilities:** Information not available in the provided data.
* **Total Debt:**
  * **2025-09-30:** 9.865700e+10 ($98.657 billion)
  * **2024-09-30:** 1.066290e+11 ($106.629 billion)
  * **2023-09-30:** 1.110880e+11 ($111.088 billion)
  * **2022-09-30:** 1.324800e+11 ($132.480 billion)


In [ ]:
question = "Apple Balance Sheet Total Assets Total Liabilities Current Liabilities Total Non Current Liabilities"

question_embedding = embedding_model.encode([question])

distances, indices = index.search(
    question_embedding.astype("float32"),
    k=5
)

for i in indices[0]:
    print("----- Retrieved Chunk -----")
    print(chunks[i])
    print()

----- Retrieved Chunk -----
Apple Balance Sheet
                                                       2025-09-30    2024-09-30    2023-09-30    2022-09-30
Treasury Shares Number                                        NaN           NaN  0.000000e+00           NaN
Ordinary Shares Number                               1.477326e+10  1.511679e+10  1.555006e+10  1.594342e+10
Share Issued                                         1.477326e+10  1.511679e+10  1.555006e+10  1.594342e+10
Net Debt                                             6.272300e+10  7.668600e+10  8.112300e+10  9.642300e+10
Total Debt                                           9.865700e+10  1.066290e+11  1.110880e+11  1.324800e+11
Tangible Book Value                                  7.373300e+10  5.695000e+10  6.214600e+10  5.067200e+10
Invested Capital                                     1.723900e+11  1.635790e+11  1.732340e+11  1.707410e+11
Working Capital                                     -1.767400e+10 -2.340500e+10 -1.74200

In [ ]:
print("Total Assets:")
print(balance_sheet.loc["Total Assets"])

print("\nTotal Liabilities:")
print(balance_sheet.loc["Total Liabilities Net Minority Interest"])

Total Assets:
2025-09-30    3.592410e+11
2024-09-30    3.649800e+11
2023-09-30    3.525830e+11
2022-09-30    3.527550e+11
Name: Total Assets, dtype: float64

Total Liabilities:
2025-09-30    2.855080e+11
2024-09-30    3.080300e+11
2023-09-30    2.904370e+11
2022-09-30    3.020830e+11
Name: Total Liabilities Net Minority Interest, dtype: float64


In [ ]:
question = "What is Apple's operating cash flow for 2022, 2023, 2024, and 2025?"

question_embedding = embedding_model.encode([question])

distances, indices = index.search(
    question_embedding.astype("float32"),
    k=5
)

for i in indices[0]:
    print("----- Retrieved Chunk -----")
    print(chunks[i])
    print()

----- Retrieved Chunk -----
Apple Financial Metrics
          Metric        Value
      Market Cap 4.968151e+12
   Current Price 3.404200e+02
         Revenue 4.668230e+11
      Net Income 1.289300e+11
             EPS 8.720000e+00
       P/E Ratio 3.903899e+01
   Price to Book 4.625272e+01
Return on Equity 1.487510e+00
   Profit Margin 2.761900e-01
Operating Margin 3.262300e-01
  Debt to Equity 7.844500e+01
  Free Cash Flow 1.077219e+11

----- Retrieved Chunk -----
Apple Cash Flow Statement
                                                  2025-09-30    2024-09-30    2023-09-30    2022-09-30
Free Cash Flow                                  9.876700e+10  1.088070e+11  9.958400e+10  1.114430e+11
Repurchase Of Capital Stock                    -9.071100e+10 -9.494900e+10 -7.755000e+10 -8.940200e+10
Repayment Of Debt                              -1.093200e+10 -9.958000e+09 -1.115100e+10 -9.543000e+09
Issuance Of Debt                                4.481000e+09  0.000000e+00  5.228000e+09  5

In [ ]:
print("Operating Cash Flow:")
print(cash_flow.loc["Operating Cash Flow"])

Operating Cash Flow:
2025-09-30    1.114820e+11
2024-09-30    1.182540e+11
2023-09-30    1.105430e+11
2022-09-30    1.221510e+11
Name: Operating Cash Flow, dtype: float64


In [ ]:
operating_cash_flow_text = cash_flow.loc["Operating Cash Flow"].to_string()

financial_documents.append({
    "title": "Apple Operating Cash Flow",
    "content": operating_cash_flow_text
})

all_text = ""

for document in financial_documents:
    all_text += document["title"] + "\n"
    all_text += document["content"] + "\n\n"

chunks = text_splitter.split_text(all_text)

embeddings = embedding_model.encode(chunks)

dimension = embeddings.shape[1]
index = faiss.IndexFlatL2(dimension)
index.add(embeddings.astype("float32"))

print("Number of chunks:", len(chunks))
print("Number of vectors in FAISS:", index.ntotal)

Number of chunks: 24
Number of vectors in FAISS: 24


In [ ]:
question = "What is Apple's operating cash flow for 2022, 2023, 2024, and 2025?"

question_embedding = embedding_model.encode([question])

distances, indices = index.search(
    question_embedding.astype("float32"),
    k=3
)

for i in indices[0]:
    print("----- Retrieved Chunk -----")
    print(chunks[i])
    print()

----- Retrieved Chunk -----
Apple Financial Metrics
          Metric        Value
      Market Cap 4.968151e+12
   Current Price 3.404200e+02
         Revenue 4.668230e+11
      Net Income 1.289300e+11
             EPS 8.720000e+00
       P/E Ratio 3.903899e+01
   Price to Book 4.625272e+01
Return on Equity 1.487510e+00
   Profit Margin 2.761900e-01
Operating Margin 3.262300e-01
  Debt to Equity 7.844500e+01
  Free Cash Flow 1.077219e+11

Apple Operating Cash Flow
2025-09-30    1.114820e+11
2024-09-30    1.182540e+11
2023-09-30    1.105430e+11
2022-09-30    1.221510e+11

----- Retrieved Chunk -----
Apple Cash Flow Statement
                                                  2025-09-30    2024-09-30    2023-09-30    2022-09-30
Free Cash Flow                                  9.876700e+10  1.088070e+11  9.958400e+10  1.114430e+11
Repurchase Of Capital Stock                    -9.071100e+10 -9.494900e+10 -7.755000e+10 -8.940200e+10
Repayment Of Debt                              -1.093200e+1

In [ ]:
question = "What is Apple's historical stock price performance over the last five years?"

question_embedding = embedding_model.encode([question])

distances, indices = index.search(
    question_embedding.astype("float32"),
    k=3
)

for i in indices[0]:
    print("----- Retrieved Chunk -----")
    print(chunks[i])
    print()

----- Retrieved Chunk -----
Apple Historical Performance
              Open         High          Low        Close        Volume    Dividends  Stock Splits
count  1254.000000  1254.000000  1254.000000  1254.000000  1.254000e+03  1254.000000        1254.0
mean    203.315315   205.578726   201.312992   203.552578  6.335963e+07     0.003907           0.0
std      51.790739    52.277757    51.381060    51.870949  2.858368e+07     0.030764           0.0
min     123.800267   125.529397   121.992536   122.827621  1.791060e+07     0.000000           0.0
25%     163.719221   165.466764   162.345993   163.650394  4.470120e+07     0.000000           0.0
50%     189.140595   191.097134   187.890451   189.338287  5.486660e+07     0.000000           0.0
75%     235.620123   238.452066   232.682001   235.871006  7.491310e+07     0.000000           0.0
max     341.079987   345.339996   338.750000   341.070007  3.186799e+08     0.270000           0.0

----- Retrieved Chunk -----
Apple Financial Metrics

In [ ]:
first_price = historical_data["Close"].iloc[0]
last_price = historical_data["Close"].iloc[-1]

price_change = last_price - first_price
percentage_change = (price_change / first_price) * 100

print("First closing price:", round(first_price, 2))
print("Latest closing price:", round(last_price, 2))
print("Price change:", round(price_change, 2))
print("Percentage change:", round(percentage_change, 2), "%")

First closing price: 139.29
Latest closing price: 340.42
Price change: 201.13
Percentage change: 144.39 %
